# L1: Automated Project: Planning, Estimation, and Allocation

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

## Initial Imports

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
#from helper import load_env
#load_env()

import os
import yaml
from crewai import Agent, Task, Crew

<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. For more help, please see the <em>"Appendix - Tips and Help"</em> Lesson.</p>

## Set OpenAI Model

In [3]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'

## Loading Tasks and Agents YAML files

In [4]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Create Pydantic Models for Structured Output

In [5]:
from typing import List
from pydantic import BaseModel, Field

class TaskEstimate(BaseModel):
    task_name: str = Field(..., description="Name of the task")
    estimated_time_hours: float = Field(..., description="Estimated time to complete the task in hours")
    required_resources: List[str] = Field(..., description="List of resources required to complete the task")

class Milestone(BaseModel):
    milestone_name: str = Field(..., description="Name of the milestone")
    tasks: List[str] = Field(..., description="List of task IDs associated with this milestone")

class ProjectPlan(BaseModel):
    tasks: List[TaskEstimate] = Field(..., description="List of tasks with their estimates")
    milestones: List[Milestone] = Field(..., description="List of project milestones")

## Create Crew, Agents and Tasks

In [6]:
# Creating Agents
project_planning_agent = Agent(
  config=agents_config['project_planning_agent']
)

estimation_agent = Agent(
  config=agents_config['estimation_agent']
)

resource_allocation_agent = Agent(
  config=agents_config['resource_allocation_agent']
)

# Creating Tasks
task_breakdown = Task(
  config=tasks_config['task_breakdown'],
  agent=project_planning_agent
)

time_resource_estimation = Task(
  config=tasks_config['time_resource_estimation'],
  agent=estimation_agent
)

resource_allocation = Task(
  config=tasks_config['resource_allocation'],
  agent=resource_allocation_agent,
  output_pydantic=ProjectPlan # This is the structured output we want
)

# Creating Crew
crew = Crew(
  agents=[
    project_planning_agent,
    estimation_agent,
    resource_allocation_agent
  ],
  tasks=[
    task_breakdown,
    time_resource_estimation,
    resource_allocation
  ],
  verbose=True
)

/media/mike/tera_data/documents/bk_Documents_202301/miguel/learning/AI/DLAI_crewAI/self-Multi-AI-Agent-Advanced-Use-Cases-with-crewAI/.venv/lib/python3.11/site-packages/crewai_tools/tools/rag/rag_tool.py:9: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class Adapter(BaseModel, ABC):


## Crew's Inputs

In [7]:
from IPython.display import display, Markdown

project = 'Barbecue Challenge'
industry = 'Food and Entertainment'
project_objectives = 'Create a small event for people from office to participate in.'
team_members = """
- John Doe (Project Manager)
- Jane Doe (Senior Chef)
- Bob Smith (Food Supplier Expert)
- Alice Johnson (Main Assistant)
- Tom Brown (Social Media Expert)
"""
project_requirements = """
- The challenge will be to prepare an open recipe in a charcoal barbecue
- The challenge will take place in about two months from today
- The maximum number of teams that can sing on is 5
- The maximum number of people per team is 3
- The main goals for the event are to have fun, spend good time together and enjoy the food
- Create vibrant invitations for everybody in the office explaining basic rules
- Create a friendly design to be printed and to be sent through email
- Establish the timeline for signing up 
- The teams have to provide the recipes so that the company can buy all the ingredients and beverages
- Highlight some basic safety measures to be considered in order to avoid cuts, burns, etc
- The event must have a nice outdoor place, with music, bathrooms, tables and chairs for up to 20 persons
- The event details and pictures will be shared in the company's newsletter
"""

# Format the dictionary as Markdown for a better display in Jupyter Lab
formatted_output = f"""
**Project Type:** {project}

**Project Objectives:** {project_objectives}

**Industry:** {industry}

**Team Members:**
{team_members}
**Project Requirements:**
{project_requirements}
"""
# Display the formatted output as Markdown
display(Markdown(formatted_output))


**Project Type:** Barbecue Challenge

**Project Objectives:** Create a small event for people from office to participate in.

**Industry:** Food and Entertainment

**Team Members:**

- John Doe (Project Manager)
- Jane Doe (Senior Chef)
- Bob Smith (Food Supplier Expert)
- Alice Johnson (Main Assistant)
- Tom Brown (Social Media Expert)

**Project Requirements:**

- The challenge will be to prepare an open recipe in a charcoal barbecue
- The challenge will take place in about two months from today
- The maximum number of teams that can sing on is 5
- The maximum number of people per team is 3
- The main goals for the event are to have fun, spend good time together and enjoy the food
- Create vibrant invitations for everybody in the office explaining basic rules
- Create a friendly design to be printed and to be sent through email
- Establish the timeline for signing up 
- The teams have to provide the recipes so that the company can buy all the ingredients and beverages
- Highlight some basic safety measures to be considered in order to avoid cuts, burns, etc
- The event must have a nice outdoor place, with music, bathrooms, tables and chairs for up to 20 persons
- The event details and pictures will be shared in the company's newsletter



## Kicking off the crew

In [8]:
# The given Python dictionary
inputs = {
  'project_type': project,
  'project_objectives': project_objectives,
  'industry': industry,
  'team_members': team_members,
  'project_requirements': project_requirements
}

# Run the crew
result = crew.kickoff(
  inputs=inputs
)

# Agent: The Ultimate Project Planner
## Task: Carefully analyze the project_requirements for the Barbecue Challenge project and break them down into individual tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are accounted for:

- The challenge will be to prepare an open recipe in a charcoal barbecue
- The challenge will take place in about two months from today
- The maximum number of teams that can sing on is 5
- The maximum number of people per team is 3
- The main goals for the event are to have fun, spend good time together and enjoy the food
- Create vibrant invitations for everybody in the office explaining basic rules
- Create a friendly design to be printed and to be sent through email
- Establish the timeline for signing up 
- The teams have to provide the recipes so that the company can buy all the ingredients and beverages
- Highlight some basic safety measures to be considered in order to avoid cuts, burns, etc
- The ev

## Usage Metrics and Costs

Let’s see how much it would cost each time if this crew runs at scale.

In [9]:
import pandas as pd

costs = 0.150 * (crew.usage_metrics.prompt_tokens + crew.usage_metrics.completion_tokens) / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([crew.usage_metrics.dict()])
df_usage_metrics

Total costs: $0.0012


/tmp/ipykernel_12720/1074044196.py:7: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  df_usage_metrics = pd.DataFrame([crew.usage_metrics.dict()])


,total_tokens,prompt_tokens,completion_tokens,successful_requests
0,7817,4438,3379,4


## Result

In [10]:
result.pydantic.dict()

/tmp/ipykernel_12720/2853131943.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  result.pydantic.dict()


{'tasks': [{'task_name': 'Define the Challenge Scope',
   'estimated_time_hours': 2.0,
   'required_resources': ['John Doe (Project Manager)']},
  {'task_name': 'Venue Selection',
   'estimated_time_hours': 48.0,
   'required_resources': ['John Doe (Project Manager)',
    'Alice Johnson (Main Assistant)']},
  {'task_name': 'Invitations Creation',
   'estimated_time_hours': 8.0,
   'required_resources': ['Alice Johnson (Main Assistant)']},
  {'task_name': 'Team Registration Setup',
   'estimated_time_hours': 8.0,
   'required_resources': ['Alice Johnson (Main Assistant)']},
  {'task_name': 'Team Registration Period',
   'estimated_time_hours': 480.0,
   'required_resources': ['Alice Johnson (Main Assistant)']},
  {'task_name': 'Recipe Collection',
   'estimated_time_hours': 24.0,
   'required_resources': ['Jane Doe (Senior Chef)',
    'Alice Johnson (Main Assistant)']},
  {'task_name': 'Safety Guidelines Creation',
   'estimated_time_hours': 8.0,
   'required_resources': ['Jane Doe (Sen

## Inspect further

In [11]:
tasks = result.pydantic.dict()['tasks']
df_tasks = pd.DataFrame(tasks)

# Display the DataFrame as an HTML table
df_tasks.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)

/tmp/ipykernel_12720/793840085.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  tasks = result.pydantic.dict()['tasks']


,task_name,estimated_time_hours,required_resources
0,Define the Challenge Scope,2.000000,['John Doe (Project Manager)']
1,Venue Selection,48.000000,"['John Doe (Project Manager)', 'Alice Johnson (Main Assistant)']"
2,Invitations Creation,8.000000,['Alice Johnson (Main Assistant)']
3,Team Registration Setup,8.000000,['Alice Johnson (Main Assistant)']
4,Team Registration Period,480.000000,['Alice Johnson (Main Assistant)']
5,Recipe Collection,24.000000,"['Jane Doe (Senior Chef)', 'Alice Johnson (Main Assistant)']"
6,Safety Guidelines Creation,8.000000,['Jane Doe (Senior Chef)']
7,Ingredient and Beverage Sourcing,48.000000,['Bob Smith (Food Supplier Expert)']
8,Event Day Preparation,48.000000,"['Alice Johnson (Main Assistant)', '1 Extra Volunteer']"
9,Event Execution,8.000000,"['John Doe (Project Manager)', 'Jane Doe (Senior Chef)', 'Alice Johnson (Main Assistant)', '1 Extra Volunteer']"


### Inspecting Milestones

In [12]:
milestones = result.pydantic.dict()['milestones']
df_milestones = pd.DataFrame(milestones)

# Display the DataFrame as an HTML table
df_milestones.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)

/tmp/ipykernel_12720/795124587.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  milestones = result.pydantic.dict()['milestones']


,milestone_name,tasks
0,Planning Phase,"['Define the Challenge Scope', 'Venue Selection', 'Invitations Creation', 'Team Registration Setup']"
1,Registration Phase,"['Team Registration Period', 'Recipe Collection', 'Safety Guidelines Creation']"
2,Preparation Phase,"['Ingredient and Beverage Sourcing', 'Event Day Preparation']"
3,Execution Phase,['Event Execution']
4,Follow-up Phase,['Post-event Content Creation']
